In [ ]:
# Ô 1 — gói, Java, mốc giờ
import subprocess, sys, os, time
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y openjdk-8-jre-headless >/dev/null 2>&1; java -version")
sh(f"{sys.executable} -m pip install -q 'trl==0.29.1' 'transformers==5.18.0' 'peft==0.21.1' accelerate bitsandbytes pycocoevalcap sacrebleu 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
import importlib.util; assert importlib.util.find_spec("torchao") is None, "DỪNG: torchao còn"
sh(f'{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"')


In [ ]:
# Ô 2 — đường dẫn, md5, hàm chạy song song
import glob, json, shutil, hashlib, re
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
MD5 = {
 "build_branch_data.py": "619e63e123a6dbf60086e65ee94a3912",
 "ex_test_k4.jsonl": "4d8d1646402aeda72f3a826c3d663702",
 "ex_val_k4.jsonl": "3a7ae1e2360659c684234e9d6fbcb866",
 "grpo_spice.py": "07ea87b6d156d1faa391a4274dffd7cf",
 "pred_ck500_test.jsonl": "328847ac96fa3104f76cd997f4091bcd",
 "pred_ck500_vallon.jsonl": "d092faf6f862eec5db71110e6b5963e2",
 "ra_build.py": "7e4ee74274037b83d11db5d63c5e66a8",
 "ra_exemplars.py": "319857d87a6d5fdf8578886ffbb1ac0f",
 "ra_gen.py": "d3969f6105b53b9eae81e7ce521cbb27",
 "ra_score.py": "04aab5e14ee01468b2c6fb8683bc2f8a",
 "ra_train.py": "217c75f188046c4bd279bb11fb1757e4",
 "train_tru_val.jsonl": "edbb231a58574acf6e1968a44d1fa6c6"
}
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ra_train.py", recursive=True)
       if os.path.exists(os.path.join(os.path.dirname(p), "train_tru_val.jsonl"))]
assert len(SRC) == 1, f"DỪNG: cần đúng một dataset ra-sft-script, thấy {SRC}"
SRC = SRC[0]
for f, h in MD5.items():
    assert md5(f"{SRC}/{f}") == h, f"DỪNG: {f} lệch md5"
    if f.endswith(".py"):
        shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "p1_train_rows.jsonl" in f)
CK = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
            if md5(p) == "491fa6677340393f1e4464c08a0cec98")
CK = [c for c in CK if not c.strip("/").endswith("ref")]
assert len(CK) == 1, f"DỪNG: không thấy đúng một adapter ck500, thấy {CK}"
CK500 = CK[0]
MERGED = f"{W}/s1_merged"
def chay(jobs, nhip=120):
    ps = []
    for i, (ten, cmd, gpu) in enumerate(jobs):
        if i: time.sleep(90)
        f = open(f"{W}/{ten}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
        ps.append((ten, subprocess.Popen(cmd, cwd=W, stdout=f, stderr=subprocess.STDOUT, env=env), f))
    while any(p.poll() is None for _, p, _ in ps):
        time.sleep(nhip)
        for ten, p, _ in ps:
            L = [l.strip() for l in open(f"{W}/{ten}.log", errors="ignore") if l.strip()]
            print(f" {(time.time() - T_NB) / 3600:5.2f} h · {ten} · {'xong' if p.poll() is not None else 'chạy'} · {L[-1][:110] if L else '..'}", flush=True)
    for ten, p, f in ps:
        f.close()
        print(f"--- {ten} · mã thoát {p.returncode} ---\n" + "".join(open(f"{W}/{ten}.log", errors="ignore").readlines()[-10:]), flush=True)
    return {ten: p.returncode for ten, p, _ in ps}
def hoa():
    r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED], cwd=W, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); assert r.returncode == 0, "DỪNG: hoà S1 lỗi"
print("SRC   ", SRC); print("BUNDLE", BUNDLE); print("CK500 ", CK500, flush=True)


In [ ]:
# Ô hoà S1
hoa()
assert os.path.isdir(MERGED)

In [ ]:
# Ô C1-1: chép mã, hàm chạy nền
import os, json, time, shutil, subprocess, glob
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"

def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g:
            return g[0]
    raise FileNotFoundError(duoi)

CW = os.path.dirname(tim("cw_gen.py"))
shutil.copytree(SRC, RA, dirs_exist_ok=True)
for f in os.listdir(CW):
    shutil.copy(f"{CW}/{f}", RA)
print("CW", CW, sorted(os.listdir(RA)))
T0 = T_NB  # tính giờ từ đầu notebook (gồm pip + hoà S1) để chắc không chạm trần 12 h

def chay(script, args, log, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"),
                            stderr=subprocess.STDOUT, env=env, cwd=RA, start_new_session=True)

def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120)
        print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h",
              [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]

In [ ]:
# Ô C2-1: tìm adapter CW, đặt đường dẫn tập kiểm
CWA = os.path.dirname(tim("cw_sft/final/adapter_config.json"))
PKG = "/kaggle/input/datasets/uyenle0712/thesis-score/thesis"
if not os.path.isdir(PKG):
    PKG = os.path.dirname(os.path.dirname(tim("harness/score_run.py")))
TEST = f"{PKG}/harness/dg1_cache/test_ac"
print("CWA", CWA, "\nPKG", PKG)
assert os.path.exists(f"{TEST}/test.jsonl") and os.path.exists(f"{PKG}/harness/score_run.py")

In [ ]:
# Ô C2-2: sinh CW trên val lớn rồi tập kiểm, hai GPU (~3,5 h)
def sinh(ten, recs, ocr, img, ex, expect):
    logs = [f"{W}/cw_{ten}_{i}.log" for i in (0, 1)]
    ps = [chay("cw_gen.py", ["--mode", "cw", "--merged", M, "--ckpt", CWA, "--recs", recs, "--ocr", ocr,
                              "--images", img, "--ra", ex, "--chi-cham", "--expect", str(expect),
                              "--out", f"{W}/cw_{ten}_{i}.jsonl", "--shard", f"{i}/2"], logs[i], i) for i in (0, 1)]
    doi(ps, logs)
    with open(f"{W}/cw_{ten}.jsonl", "w") as f:
        for i in (0, 1):
            f.write(open(f"{W}/cw_{ten}_{i}.jsonl").read())
    print(open(logs[0]).read()[:3000])

sinh("val", f"{BUNDLE}/p1_val_rows.jsonl", f"{BUNDLE}/ocr.jsonl", f"{BUNDLE}/images", f"{SRC}/ex_val_k4.jsonl", 1002)
sinh("test", f"{TEST}/test.jsonl", f"{TEST}/ocr.jsonl", f"{TEST}/images", f"{SRC}/ex_test_k4.jsonl", 4463)

In [ ]:
# Ô C2-3: kiểm đầu ra của thẻ trước khi chấm
import math
for ten, n in (("val", 1002), ("test", 4463)):
    C = [json.loads(l) for l in open(f"{W}/cw_{ten}.jsonl")]
    assert len(C) == n, (ten, len(C))
    assert all(all(math.isfinite(x) and x <= 0 for x in c["lp"]) for c in C), "⛔ lp không hợp lệ"
    pc = sorted(c["p_copy"] for c in C)
    print(ten, "p_copy trung vị", round(pc[n // 2], 3), "- tỉ lệ ≥ τ:",
          {t: round(100 * sum(x >= t for x in pc) / n, 1) for t in (0.3, 0.5, 0.7, 0.9)},
          "- câu viết rỗng", sum(not c["write"] for c in C))

In [ ]:
# Ô C2-4: cài bộ chấm — theo ô cài của uyenle0712/ra-score-test-304 (bản đã chấm ra đúng số ck500)
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y openjdk-8-jre-headless >/dev/null 2>&1; update-alternatives --set java $(update-alternatives --list java | grep java-8 | head -1); java -version")
sh(f"{sys.executable} -m pip install -q pycocoevalcap sacrebleu bert_score 2>&1 | tail -3")

In [ ]:
# Ô C2-5: ghép lưới τ trên val, chấm 5 thước, CHỐT τ một lần
def py(*args):
    r = subprocess.run(["python", *args], capture_output=True, text=True, cwd=RA)
    print(r.stdout[-5000:], r.stderr[-2000:]); assert r.returncode == 0

py(f"{RA}/cw_ghep.py", "--cw", f"{W}/cw_val.jsonl", "--ra", f"{SRC}/ex_val_k4.jsonl", "--out-dir", f"{W}/ghep", "--ten", "val")
LUOI = ("0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.01")
py(f"{RA}/ra_score.py", "--recs", f"{BUNDLE}/p1_val_rows.jsonl", "--pred", f"ck500={SRC}/pred_ck500_vallon.jsonl",
   *sum([["--pred", f"cw_t{t}={W}/ghep/pred_cw_t{t}_val.jsonl"] for t in LUOI], []),
   "--no-spice", "--no-bert", "--out", f"{W}/diem_val_cw.json")
py(f"{RA}/cw_chon_tau.py", "--diem", f"{W}/diem_val_cw.json", "--out", f"{W}/tau_chon.json")

In [ ]:
# Ô C2-6: áp τ đã chốt lên tập kiểm MỘT lần, chấm đủ 7 thước + tỉ lệ chép
tau = json.load(open(f"{W}/tau_chon.json"))["tau"]
for t in (tau, 1.01):
    py(f"{RA}/cw_ghep.py", "--cw", f"{W}/cw_test.jsonl", "--ra", f"{SRC}/ex_test_k4.jsonl",
       "--out-dir", f"{W}/ghep", "--ten", "test", "--tau", str(t))
ten_ra = [f"cw_t{tau}"] if tau == 1.01 else [f"cw_t{tau}", "cw_t1.01"]
try:
    py(f"{RA}/ra_score.py", "--recs", f"{TEST}/test.jsonl", "--pred", f"ck500={SRC}/pred_ck500_test.jsonl",
       *sum([["--pred", f"{n}={W}/ghep/pred_{n}_test.jsonl"] for n in ten_ra], []),
       "--ra", f"cw_t{tau}={SRC}/ex_test_k4.jsonl", "--check", "--out", f"{W}/diem_test_cw.json")
except AssertionError:
    print("⚠️ chấm 7 thước tập kiểm lỗi — vẫn chạy exec + zip; chấm lại riêng sau từ ghep/pred_*_test.jsonl", flush=True)

In [ ]:
# Ô C2-7: exec — chỉ chấm UGround các câu cw_t{τ*} khác ck500 + 40 câu ck500 không đổi để hiệu chuẩn (~30–60 phút)
import random
E = f"{W}/exec_out"; os.makedirs(E, exist_ok=True)
ck = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{SRC}/pred_ck500_test.jsonl"))}
cw = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{W}/ghep/pred_cw_t{tau}_test.jsonl"))}
doi_k = sorted(k for k in cw if cw[k] != ck[k])
cal = random.Random(20261010).sample(sorted(k for k in cw if cw[k] == ck[k]), 40)
print(f"câu khác ck500: {len(doi_k)} · hiệu chuẩn: {len(cal)}")

def ghi(p, ks, P):
    with open(p, "w") as f:
        for k in ks:
            f.write(json.dumps({"episode_id": k[0], "step_id": k[1], "pred": P[k]}, ensure_ascii=False) + "\n")

for i in (0, 1):
    ghi(f"{E}/cal_{i}.jsonl", cal[i::2], ck)
    ghi(f"{E}/cw_{i}.jsonl", doi_k[i::2], cw)

WS = f"{W}/thesis"
shutil.copytree(f"{PKG}/harness", f"{WS}/harness", dirs_exist_ok=True, ignore=shutil.ignore_patterns("images"))
link = f"{WS}/harness/dg1_cache/test_ac/images"
if os.path.lexists(link): os.remove(link)
os.symlink(f"{TEST}/images", link)

def cham(ten, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TQDM_DISABLE="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
    return subprocess.Popen(["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground",
                             "--preds", f"{E}/{ten}.jsonl", "--out", f"{E}/score_{ten}.json", "--n", "99999"],
                            stdout=open(f"{E}/score_{ten}.log", "w"),
                            stderr=subprocess.STDOUT, env=env, cwd=WS, start_new_session=True)

try:
    for loai in ("cal", "cw"):
        ps = [cham(f"{loai}_{i}", i) for i in (0, 1)]
        doi(ps, [f"{E}/score_{loai}_{i}.log" for i in (0, 1)])
finally:
    print(sorted(os.listdir(E)))
    shutil.rmtree(WS, ignore_errors=True)
for d in ("s1_merged", "ra"):
    shutil.rmtree(f"{W}/{d}", ignore_errors=True)
shutil.make_archive("/tmp/ket_qua_309_G", "zip", W, ".")
shutil.move("/tmp/ket_qua_309_G.zip", f"{W}/ket_qua_309_G.zip")
print(f"{(time.time()-T0)/3600:.2f} h")